In [20]:
# 1. Install torchinfo (only needed once per notebook session)
!pip install torchinfo -q

import torch
import torch.nn as nn
from torchinfo import summary

# 2. Define the Architecture (The Blueprint)
class ScratchCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ScratchCNN, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    # 3. Define the Flow (Forward Pass)
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x

# 4. Instantiate the model and run the X-Ray!
model = ScratchCNN(num_classes=4)

print("="*70)
print(" SCRATCH CNN VERIFICATION (Matching Hand Calculations)")
print("="*70)

# input_size=(1, 3, 224, 224) means: 1 image, 3 channels (RGB), 224x224 pixels
summary(model, input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "num_params"])

 SCRATCH CNN VERIFICATION (Matching Hand Calculations)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
ScratchCNN                               [1, 3, 224, 224]          [1, 4]                    --
├─Sequential: 1-1                        [1, 3, 224, 224]          [1, 16, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 3, 224, 224]          [1, 16, 224, 224]         448
│    └─BatchNorm2d: 2-2                  [1, 16, 224, 224]         [1, 16, 224, 224]         32
│    └─ReLU: 2-3                         [1, 16, 224, 224]         [1, 16, 224, 224]         --
│    └─MaxPool2d: 2-4                    [1, 16, 224, 224]         [1, 16, 112, 112]         --
├─Sequential: 1-2                        [1, 16, 112, 112]         [1, 32, 56, 56]           --
│    └─Conv2d: 2-5                       [1, 16, 112, 112]         [1, 32, 112, 112]         4,640
│    └─BatchNorm2d: 2-6                  [1, 32, 112, 112]         [1, 32, 112, 112]         64
│    └─ReLU: 2-7               

In [21]:
import torch
import torch.nn.functional as F

# 1. The Setup (Exactly like our hand calculation)
# 4x4 Input of 1s
X = torch.ones(1, 1, 4, 4) 

# 3x3 Filter of 1s. requires_grad=True tells PyTorch to track the math for this!
W = torch.ones(1, 1, 3, 3, requires_grad=True) 

# Target Output of 0s
Y = torch.zeros(1, 1, 2, 2) 

# 2. Forward Pass
output = F.conv2d(X, W)

# 3. Calculate Loss
# IMPORTANT: We use reduction='sum' to match our hand calculation exactly. 
# (If we used 'mean', PyTorch would divide by 4, and the gradient would be 18 instead of 72).
loss = F.mse_loss(output, Y, reduction='sum') 

# 4. Backward Pass (The Magic)
loss.backward()

# 5. Check the Gradient
print("Hand Calculated Gradient: 72")
print("PyTorch Autograd Gradient:\n", W.grad)

Hand Calculated Gradient: 72
PyTorch Autograd Gradient:
 tensor([[[[72., 72., 72.],
          [72., 72., 72.],
          [72., 72., 72.]]]])


In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. CONFIGURATION FOR OVERFIT TEST
BATCH_SIZE = 16
EPOCHS = 50  
LEARNING_RATE = 0.001

# 2. LOAD TINY DATASET (Only 100 images)
BRISC_HF_ID = "srijan2715/brisc2025-paper-pipeline"
dataset = load_dataset(BRISC_HF_ID)
tiny_dataset = dataset['train'].select(range(100))

# CLASS MAPPING
CLASS_TO_IDX = {
    "glioma": 0,
    "meningioma": 1,
    "no_tumor": 2,
    "pituitary": 3
}

def final_transform(data):
    # --- HANDLE IMAGES ---
    if isinstance(data['image'], list):
        # Batch case: list of 3D lists
        images = []
        for img_3d in data['image']:
            # Convert 3D list [224][224][3] to tensor, then permute to [3][224][224]
            img_tensor = torch.tensor(img_3d, dtype=torch.float32)  # Shape: [224, 224, 3]
            img_tensor = img_tensor.permute(2, 0, 1)  # Shape: [3, 224, 224]
            images.append(img_tensor / 255.0)
        data['image'] = torch.stack(images)
    else:
        # Single case
        img_tensor = torch.tensor(data['image'], dtype=torch.float32)  # Shape: [224, 224, 3]
        img_tensor = img_tensor.permute(2, 0, 1)  # Shape: [3, 224, 224]
        data['image'] = img_tensor / 255.0
    
    # --- HANDLE LABELS ---
    def str_to_int(lbl):
        lbl_str = str(lbl).lower().replace(" ", "_")
        return CLASS_TO_IDX.get(lbl_str, 0)
        
    if isinstance(data['label'], list):
        data['label'] = torch.tensor([str_to_int(x) for x in data['label']], dtype=torch.long)
    else:
        data['label'] = torch.tensor(str_to_int(data['label']), dtype=torch.long)
        
    return data

# Apply the transform
tiny_dataset = tiny_dataset.with_transform(final_transform)

# IMPORTANT: num_workers=0 for debugging
tiny_loader = DataLoader(tiny_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 3. TRAINING LOOP
print("\n STARTING 100-IMAGE OVERFIT TEST (FINAL CORRECTED VERSION)...")
model = ScratchCNN(num_classes=4).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch in tiny_loader:
        images, labels = batch['image'].to(DEVICE), batch['label'].to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    avg_loss = running_loss / len(tiny_loader)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {train_acc:.2f}%")

print("\n✅ OVERFIT TEST COMPLETE!")
print(f"Final Loss: {avg_loss:.4f} | Final Accuracy: {train_acc:.2f}%")

Using device: cpu

 STARTING 100-IMAGE OVERFIT TEST (FINAL CORRECTED VERSION)...
Epoch  1/50 | Loss: 1.2596 | Acc: 49.00%
Epoch 10/50 | Loss: 0.7714 | Acc: 73.00%
Epoch 20/50 | Loss: 0.5712 | Acc: 80.00%
Epoch 30/50 | Loss: 0.3361 | Acc: 94.00%
Epoch 40/50 | Loss: 0.2684 | Acc: 93.00%
Epoch 50/50 | Loss: 0.1897 | Acc: 99.00%

✅ OVERFIT TEST COMPLETE!
Final Loss: 0.1897 | Final Accuracy: 99.00%


In [23]:
from datasets import load_dataset

BRISC_HF_ID = "srijan2715/brisc2025-paper-pipeline"
dataset = load_dataset(BRISC_HF_ID)
item = dataset['train'][0]
img_data = item['image']

print("Structure Analysis:")
print(f"1. Outer list length (rows): {len(img_data)}")
print(f"2. Inner list length (cols): {len(img_data[0])}")
print(f"3. Type of img_data[0][0]: {type(img_data[0][0])}")

# Check what's inside the innermost element
if isinstance(img_data[0][0], list):
    print(f"4. Innermost is a list of length: {len(img_data[0][0])}")
    print(f"5. Sample pixel value (RGB): {img_data[0][0]}")
    print(f"6. This is a 3D structure: [{len(img_data)}][{len(img_data[0])}][{len(img_data[0][0])}]")
else:
    print(f"4. Innermost is a single value: {img_data[0][0]}")
    print(f"5. This is a 2D structure (grayscale): [{len(img_data)}][{len(img_data[0])}]")

print(f"\n7. Label: {item['label']}")

Structure Analysis:
1. Outer list length (rows): 224
2. Inner list length (cols): 224
3. Type of img_data[0][0]: <class 'list'>
4. Innermost is a list of length: 3
5. Sample pixel value (RGB): [1, 1, 1]
6. This is a 3D structure: [224][224][3]

7. Label: pituitary
